# MAI391 — Dataset & Preprocessing (Khoa)
**Project:** Wine Quality Prediction Using Multiple Linear Regression  
**Nguồn:** UCI Wine Quality — Red Wine  
**Mục tiêu:** kiểm tra dữ liệu, thống kê mô tả, chuẩn bị `X`, `y`, chia train/test 80/20 và tạo phương án StandardScaler tùy chọn.  
**Lưu ý:** Baseline **giữ nguyên các dòng trùng**; không train model trong notebook của Khoa. EDA plots do Thao phụ trách.

Run **Runtime → Run all** trên Google Colab. Notebook xuất kết quả vào thư mục `MAI_Khoa_outputs` và nén lại thành ZIP để bàn giao.

## 1. Import thư viện và cấu hình chung
Giữ cố định `random_state=42`; đọc CSV với dấu phân tách `;`.

In [17]:
from pathlib import Path
import hashlib
import json
import shutil
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

RANDOM_STATE = 42
TEST_SIZE = 0.20
SOURCE_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
OUT = Path('MAI_Khoa_outputs')
OUT.mkdir(exist_ok=True)
print('pandas:', pd.__version__)
print('Outputs:', OUT.resolve())

pandas: 2.2.3
Outputs: /content/MAI_Khoa_outputs


## 2. Tải file gốc và đọc bằng Pandas
Nếu Colab không truy cập được UCI, hãy tải file `winequality-red.csv` bằng trình duyệt và upload lên Colab. Notebook sẽ ưu tiên file cục bộ khi có sẵn. **Không** thay bằng tập wine khác.

In [35]:
local_csv = Path('winequality-red.csv')
if local_csv.exists():
    print('Đọc CSV cục bộ:', local_csv)
    data_bytes = local_csv.read_bytes()
else:
    import urllib.request
    try:
        with urllib.request.urlopen(SOURCE_URL, timeout=30) as response:
            data_bytes = response.read()
        local_csv.write_bytes(data_bytes)
        print('Đã tải dữ liệu từ UCI:', local_csv)
    except Exception as exc:
        raise RuntimeError(
            'Không tải được CSV từ UCI. Hãy upload file winequality-red.csv lên Colab '
            'rồi chạy lại notebook. URL: ' + SOURCE_URL
        ) from exc

sha256 = hashlib.sha256(data_bytes).hexdigest()
df = pd.read_csv(local_csv, sep=';')
print('Shape:', df.shape)
print('SHA-256:', sha256)
display(df.head())


# 1. Kiểm tra kích thước
print("=== DATASET SHAPE ===")
print(df.shape)

# 2. Kiểm tra kiểu dữ liệu
print("\n=== DATA TYPES ===")
df.info()

# 3. Kiểm tra dữ liệu thiếu
print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

# 4. Kiểm tra dòng trùng lặp
print("\n=== DUPLICATE ROWS ===")
print(df.duplicated().sum())

# 5. Thống kê mô tả
print("\n=== DESCRIPTIVE STATISTICS ===")
display(df.describe())


# BƯỚC 5: CHUẨN BỊ FEATURES VÀ TARGET

# X gồm 11 đặc trưng đầu vào
X = df.drop(columns=["quality"])

# y là biến mục tiêu
y = df["quality"]

print("=== FEATURES (X) ===")
print("Kích thước X:", X.shape)
print("Danh sách features:")
print(X.columns.tolist())

print("\n=== TARGET (y) ===")
print("Kích thước y:", y.shape)
print("Tên target:", y.name)

print("\n5 dòng đầu tiên của X:")
display(X.head())

print("\n5 giá trị đầu tiên của y:")
display(y.head())


# BƯỚC 6: CHIA DỮ LIỆU TRAIN/TEST

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("=== TRAIN/TEST SPLIT ===")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))


# BƯỚC 7: CHUẨN HÓA DỮ LIỆU

from sklearn.preprocessing import StandardScaler
import pandas as pd

# Khởi tạo StandardScaler
scaler = StandardScaler()

# Fit trên train, sau đó transform train
X_train_scaled = scaler.fit_transform(X_train)

# Chỉ transform test
X_test_scaled = scaler.transform(X_test)

# Chuyển về DataFrame để dễ quan sát
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

print("=== STANDARD SCALER ===")
print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)

print("\n5 dòng đầu tiên sau chuẩn hóa:")
display(X_train_scaled.head())

print("\nMean của tập train sau chuẩn hóa:")
print(X_train_scaled.mean().round(6))


# BƯỚC 8: XUẤT DỮ LIỆU

from pathlib import Path
import json
import pandas as pd

# Tạo thư mục lưu kết quả
output_dir = Path("Khoa_Outputs")
output_dir.mkdir(exist_ok=True)

# Xuất train/test, giữ cả 11 features và quality
train_df = X_train.copy()
train_df["quality"] = y_train

test_df = X_test.copy()
test_df["quality"] = y_test

train_df.to_csv(output_dir / "train.csv", index=False)
test_df.to_csv(output_dir / "test.csv", index=False)

# Xuất thống kê mô tả
df.describe().to_csv(
    output_dir / "descriptive_statistics.csv"
)

# Xuất thông tin kiểm tra dữ liệu
report = {
    "dataset_shape": list(df.shape),
    "missing_values": int(df.isnull().sum().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
    "train_samples": len(X_train),
    "test_samples": len(X_test),
    "random_state": 42,
    "feature_order": X.columns.tolist(),
    "target": "quality",
    "duplicates_removed": False
}

with open(output_dir / "inspection_report.json",
          "w", encoding="utf-8") as f:
    json.dump(report, f, indent=4, ensure_ascii=False)

# Xuất phân phối quality cho Thao
df["quality"].value_counts().sort_index().to_csv(
    output_dir / "quality_distribution.csv",
    header=["count"]
)

print("Đã xuất dữ liệu thành công!")
print("Thư mục:", output_dir)

for file in output_dir.iterdir():
    print("-", file.name)


import shutil
from google.colab import files

shutil.make_archive(
    "Khoa_Outputs",
    "zip",
    "Khoa_Outputs"
)

files.download("Khoa_Outputs.zip")


# BƯỚC 8.1: XUẤT KẾT QUẢ PREPROCESSING

import pandas as pd
import json
import joblib
from pathlib import Path

# Tạo thư mục lưu kết quả
output_dir = Path("Khoa_Outputs")
output_dir.mkdir(exist_ok=True)

# 1. Lưu dữ liệu Train/Test gốc
train_df = X_train.copy()
train_df["quality"] = y_train

test_df = X_test.copy()
test_df["quality"] = y_test

train_df.to_csv(output_dir / "train.csv", index=False)
test_df.to_csv(output_dir / "test.csv", index=False)

# 2. Lưu thống kê mô tả
df.describe().to_csv(
    output_dir / "descriptive_statistics.csv"
)

# 3. Lưu phân phối quality cho Thao
df["quality"].value_counts().sort_index().to_csv(
    output_dir / "quality_distribution.csv",
    header=["count"]
)

# 4. Lưu báo cáo kiểm tra dữ liệu
report = {
    "dataset_shape": list(df.shape),
    "missing_values": int(df.isnull().sum().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
    "train_samples": len(X_train),
    "test_samples": len(X_test),
    "test_size": 0.2,
    "random_state": 42,
    "feature_order": X.columns.tolist(),
    "target": "quality",
    "duplicates_removed": False
}

with open(output_dir / "inspection_report.json",
          "w", encoding="utf-8") as f:
    json.dump(report, f, indent=4, ensure_ascii=False)

# 5. Lưu StandardScaler đã fit trên train
joblib.dump(scaler, output_dir / "standard_scaler.pkl")

print("=== XUẤT FILE THÀNH CÔNG ===")

for file in sorted(output_dir.iterdir()):
    print(file.name)


# BƯỚC 8.2: NÉN VÀ TẢI KẾT QUẢ

import shutil
from google.colab import files

shutil.make_archive(
    "Khoa_Outputs",
    "zip",
    "Khoa_Outputs"
)

files.download("Khoa_Outputs.zip")


Đọc CSV cục bộ: winequality-red.csv
Shape: (1599, 12)
SHA-256: 4a402cf041b025d4566d954c3b9ba8635a3a8a01e039005d97d6a710278cf05e


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


=== DATASET SHAPE ===
(1599, 12)

=== DATA TYPES ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB

=== MISSING VALUES ===
fixed acidity           0
volatile acidity       

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000
mean,8.319637,0.527821,0.270976,2.538806,0.087467,15.874922,46.467792,0.996747,3.311113,0.658149,10.422983,5.636023
std,1.741096,0.179060,0.194801,1.409928,0.047065,10.460157,32.895324,0.001887,0.154386,0.169507,1.065668,0.807569
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.330000,8.400000,3.000000
25%,7.100000,0.390000,0.090000,1.900000,0.070000,7.000000,22.000000,0.995600,3.210000,0.550000,9.500000,5.000000
50%,7.900000,0.520000,0.260000,2.200000,0.079000,14.000000,38.000000,0.996750,3.310000,0.620000,10.200000,6.000000
75%,9.200000,0.640000,0.420000,2.600000,0.090000,21.000000,62.000000,0.997835,3.400000,0.730000,11.100000,6.000000
max,15.900000,1.580000,1.000000,15.500000,0.611000,72.000000,289.000000,1.003690,4.010000,2.000000,14.900000,8.000000


=== FEATURES (X) ===
Kích thước X: (1599, 11)
Danh sách features:
['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']

=== TARGET (y) ===
Kích thước y: (1599,)
Tên target: quality

5 dòng đầu tiên của X:


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4


... (6 outputs left out)

<IPython.core.display.Javascript object>

=== XUẤT FILE THÀNH CÔNG ===
descriptive_statistics.csv
inspection_report.json
quality_distribution.csv
standard_scaler.pkl
test.csv
train.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 3. Kiểm tra schema, dữ liệu thiếu, dữ liệu trùng, kiểu dữ liệu

In [19]:
EXPECTED_COLUMNS = [
    'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
    'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
    'pH', 'sulphates', 'alcohol', 'quality'
]
assert list(df.columns) == EXPECTED_COLUMNS, 'Sai tên/thứ tự cột. Kiểm tra file CSV gốc và sep=";".'
assert df.shape == (1599, 12), 'Không đúng bản UCI Red Wine Quality chuẩn.'
assert all(pd.api.types.is_numeric_dtype(df[c]) for c in df.columns), 'Có cột không phải dạng số.'

missing = df.isna().sum().rename('missing_count')
dtype_table = df.dtypes.astype(str).rename('dtype')
report_by_column = pd.concat([dtype_table, missing], axis=1)
report_by_column['missing_pct'] = (report_by_column['missing_count'] / len(df) * 100).round(2)
duplicate_count = int(df.duplicated().sum())
unique_row_count = int(len(df.drop_duplicates()))
print('Missing cells:', int(df.isna().sum().sum()))
print('Duplicate rows (bỏ dòng xuất hiện đầu tiên):', duplicate_count)
print('Unique rows:', unique_row_count)
display(report_by_column)
report_by_column.to_csv(OUT / 'column_quality_report.csv', encoding='utf-8-sig')

Missing cells: 0
Duplicate rows (bỏ dòng xuất hiện đầu tiên): 240
Unique rows: 1359


,dtype,missing_count,missing_pct
fixed acidity,float64,0,0.0
volatile acidity,float64,0,0.0
citric acid,float64,0,0.0
residual sugar,float64,0,0.0
chlorides,float64,0,0.0
free sulfur dioxide,float64,0,0.0
total sulfur dioxide,float64,0,0.0
density,float64,0,0.0
pH,float64,0,0.0
sulphates,float64,0,0.0


**Chính sách xử lý:** Báo cáo đầy đủ dòng trùng nhưng **không gọi `drop_duplicates()`** để bảo toàn 1.599 mẫu trong baseline nhóm đã thống nhất. Các dòng giống hệt nhau có thể xuất hiện ở cả train và test; đây là hạn chế phương pháp cần nêu. Nếu nhóm đổi sang xóa trùng / chia theo nhóm trùng thì **mọi thành viên phải chạy lại từ cùng một quy trình**.

## 4. Thống kê mô tả và kiểm tra điểm chất lượng

In [20]:
summary_stats = df.describe().T
summary_stats.to_csv(OUT / 'descriptive_statistics.csv', encoding='utf-8-sig')
quality_distribution = (df['quality'].value_counts().sort_index()
                        .rename_axis('quality').reset_index(name='count'))
quality_distribution['percentage'] = (quality_distribution['count'] / len(df) * 100).round(2)
quality_distribution.to_csv(OUT / 'quality_distribution.csv', index=False, encoding='utf-8-sig')
print('Thống kê mô tả:')
display(summary_stats.round(4))
print('Phân phối biến mục tiêu:')
display(quality_distribution)
print('Giá trị quality quan sát:', sorted(df['quality'].unique().tolist()))
print('Quality min/max:', df['quality'].min(), df['quality'].max())

Thống kê mô tả:


,count,mean,std,min,25%,50%,75%,max
fixed acidity,1599.0,8.3196,1.7411,4.6000,7.1000,7.9000,9.2000,15.9000
volatile acidity,1599.0,0.5278,0.1791,0.1200,0.3900,0.5200,0.6400,1.5800
citric acid,1599.0,0.2710,0.1948,0.0000,0.0900,0.2600,0.4200,1.0000
residual sugar,1599.0,2.5388,1.4099,0.9000,1.9000,2.2000,2.6000,15.5000
chlorides,1599.0,0.0875,0.0471,0.0120,0.0700,0.0790,0.0900,0.6110
free sulfur dioxide,1599.0,15.8749,10.4602,1.0000,7.0000,14.0000,21.0000,72.0000
total sulfur dioxide,1599.0,46.4678,32.8953,6.0000,22.0000,38.0000,62.0000,289.0000
density,1599.0,0.9967,0.0019,0.9901,0.9956,0.9968,0.9978,1.0037
pH,1599.0,3.3111,0.1544,2.7400,3.2100,3.3100,3.4000,4.0100
sulphates,1599.0,0.6581,0.1695,0.3300,0.5500,0.6200,0.7300,2.0000


Phân phối biến mục tiêu:


,quality,count,percentage
0,3,10,0.63
1,4,53,3.31
2,5,681,42.59
3,6,638,39.90
4,7,199,12.45
5,8,18,1.13


Giá trị quality quan sát: [3, 4, 5, 6, 7, 8]
Quality min/max: 3 8
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


## 5. Xác định X và y — cố định thứ tự 11 đặc trưng

In [21]:
FEATURE_COLUMNS = EXPECTED_COLUMNS[:-1]
TARGET_COLUMN = 'quality'
X = df.loc[:, FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()
assert X.shape == (1599, 11) and y.shape == (1599,)
assert TARGET_COLUMN not in X.columns
print('X:', X.shape, '| y:', y.shape)
for i, feature in enumerate(FEATURE_COLUMNS, start=1):
    print(f'{i:2}. {feature}')
(OUT / 'feature_order.json').write_text(
    json.dumps({'features': FEATURE_COLUMNS, 'target': TARGET_COLUMN}, indent=2, ensure_ascii=False),
    encoding='utf-8')

X: (1599, 11) | y: (1599,)
 1. fixed acidity
 2. volatile acidity
 3. citric acid
 4. residual sugar
 5. chlorides
 6. free sulfur dioxide
 7. total sulfur dioxide
 8. density
 9. pH
10. sulphates
11. alcohol


260

## 6. Chia train/test 80/20 (random_state = 42)
**Chỉ chia một lần.** Cung cấp index file cho cả nhóm; không chia lại trong các notebook độc lập. Không stratify để giữ đúng cùng lời gọi `train_test_split` tiêu chuẩn.

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
assert len(X_train) == 1279 and len(X_test) == 320
assert X_train.index.intersection(X_test.index).empty
print('X_train:', X_train.shape, 'X_test:', X_test.shape)
print('y_train:', y_train.shape, 'y_test:', y_test.shape)
print('Tỉ lệ:', round(len(X_train)/len(df)*100, 2), '/', round(len(X_test)/len(df)*100, 2))

# Giữ index gốc trong file riêng để có thể tái lập và đối chiếu
pd.Series(X_train.index, name='original_row_index').to_csv(OUT / 'train_indices.csv', index=False)
pd.Series(X_test.index, name='original_row_index').to_csv(OUT / 'test_indices.csv', index=False)
X_train.to_csv(OUT / 'X_train.csv', index=False)
X_test.to_csv(OUT / 'X_test.csv', index=False)
y_train.to_frame().to_csv(OUT / 'y_train.csv', index=False)
y_test.to_frame().to_csv(OUT / 'y_test.csv', index=False)

# Gộp dataset train/test để thành viên 4 có thể load tiện hơn
X_train.assign(quality=y_train).to_csv(OUT / 'train.csv', index=False)
X_test.assign(quality=y_test).to_csv(OUT / 'test.csv', index=False)

# Check exact duplicates across train/test by row content; note possibility of overlap
train_rows = pd.MultiIndex.from_frame(X_train.assign(quality=y_train))
test_rows = pd.MultiIndex.from_frame(X_test.assign(quality=y_test))
overlap_unique = len(set(train_rows).intersection(set(test_rows)))
print('Unique full-record values shared between train & test:', overlap_unique)
print('Giữ duplicates theo kế hoạch baseline; cần ghi hạn chế này trong báo cáo.')

X_train: (1279, 11) X_test: (320, 11)
y_train: (1279,) y_test: (320,)
Tỉ lệ: 79.99 / 20.01
Unique full-record values shared between train & test: 62
Giữ duplicates theo kế hoạch baseline; cần ghi hạn chế này trong báo cáo.


## 7. StandardScaler — phương án tùy chọn, KHÔNG dùng mặc định
Pipeline bảo đảm scaler chỉ **fit trên X_train**. Khi gọi `predict(X_test)`, pipeline dùng chính scaler đã học để `transform` test, tránh leakage. M4 chỉ bật nếu nhóm thống nhất muốn thử chuẩn hóa.

In [23]:
# OPTION A (baseline M4): sử dụng trực tiếp X_train, X_test chưa scale.
# OPTION B (tùy chọn): dùng pipeline sau, không cần tự scale trước:
optional_scaled_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
# M4 có thể gọi:
# optional_scaled_pipeline.fit(X_train, y_train)
# y_pred = optional_scaled_pipeline.predict(X_test)

# Nếu cần lấy ma trận đã scale (không fit trên test):
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train),
                              columns=FEATURE_COLUMNS, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test),
                             columns=FEATURE_COLUMNS, index=X_test.index)
print('Mean of scaled train (gần 0):', X_train_scaled.mean().abs().max())
print('Lưu ý: không xuất file scaled riêng vì đã bàn giao code/pipeline.')

Mean of scaled train (gần 0): 2.559398267120236e-14
Lưu ý: không xuất file scaled riêng vì đã bàn giao code/pipeline.


## 8. Xuất manifest, báo cáo ngắn và nén file bàn giao

In [24]:
manifest = {
    'dataset': 'UCI Wine Quality — Red', 'source_url': SOURCE_URL,
    'source_sha256': sha256,
    'shape': list(df.shape), 'features': FEATURE_COLUMNS, 'target': TARGET_COLUMN,
    'missing_cells': int(df.isna().sum().sum()), 'duplicate_rows': duplicate_count,
    'unique_rows': unique_row_count,
    'split': {'test_size': TEST_SIZE, 'random_state': RANDOM_STATE,
              'shuffle': True, 'stratify': None,
              'train_size': len(X_train), 'test_size_rows': len(X_test)},
    'duplicate_policy': 'report but retain, original dataset used for baseline',
    'unique_full_record_values_in_both_sets': overlap_unique,
    'scaling': 'optional; fit on train only; transform on test',
}
(OUT / 'preprocessing_manifest.json').write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
report_text = f'''# Kết quả kiểm tra dữ liệu — Khoa

- Nguồn: {SOURCE_URL}
- File SHA256: `{sha256}`
- Kích thước: **{df.shape[0]} dòng × {df.shape[1]} cột**.
- Số features: **{X.shape[1]}**; target: **`quality`**.
- Missing values: **{int(df.isna().sum().sum())}**.
- Duplicated rows (theo pandas): **{duplicate_count}**; giữ nguyên trong baseline.
- Train: **{len(X_train)}**; Test: **{len(X_test)}**.
- Split: `test_size=0.2`, `random_state=42`.
- Mức quality quan sát: **{sorted(df.quality.unique().tolist())}**.
- Số tổ hợp giá trị bản ghi giống hệt nhau có ở cả train và test: **{overlap_unique}**.
- Dữ liệu đầu vào giữ nguyên thang đo; StandardScaler chỉ là lựa chọn thử nghiệm.

## Bàn giao
Cung cấp cho M4: `train.csv`, `test.csv`, `feature_order.json`, `preprocessing_manifest.json`.
Cung cấp cho M2: `quality_distribution.csv`, `descriptive_statistics.csv`.
Cung cấp cho M3: `feature_order.json`.
'''
(OUT / 'inspection_report.md').write_text(report_text, encoding='utf-8')
shutil.make_archive('MAI_Khoa_outputs', 'zip', root_dir='.', base_dir=str(OUT))
print('Đã tạo:', Path('MAI_Khoa_outputs.zip').resolve())
print(report_text)

Đã tạo: /content/MAI_Khoa_outputs.zip
# Kết quả kiểm tra dữ liệu — Khoa

- Nguồn: https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv
- File SHA256: `4a402cf041b025d4566d954c3b9ba8635a3a8a01e039005d97d6a710278cf05e`
- Kích thước: **1599 dòng × 12 cột**.
- Số features: **11**; target: **`quality`**.
- Missing values: **0**.
- Duplicated rows (theo pandas): **240**; giữ nguyên trong baseline.
- Train: **1279**; Test: **320**.
- Split: `test_size=0.2`, `random_state=42`.
- Mức quality quan sát: **[3, 4, 5, 6, 7, 8]**.
- Số tổ hợp giá trị bản ghi giống hệt nhau có ở cả train và test: **62**.
- Dữ liệu đầu vào giữ nguyên thang đo; StandardScaler chỉ là lựa chọn thử nghiệm.

## Bàn giao
Cung cấp cho M4: `train.csv`, `test.csv`, `feature_order.json`, `preprocessing_manifest.json`.
Cung cấp cho M2: `quality_distribution.csv`, `descriptive_statistics.csv`.
Cung cấp cho M3: `feature_order.json`.

